In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00


In [ ]:
!pip install pyg-lib -f https://data.pyg.org/whl/torch-$(python3 -c "import torch; print(torch.__version__.split('+')[0])")+$(python3 -c "import torch; print('cu'+torch.version.cuda.replace('.','') if torch.version.cuda else 'cpu')").html

Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 51.1 MB/s eta 0:00:00


In [ ]:
!pip install torch-sparse -f https://data.pyg.org/whl/torch-$(python3 -c "import torch; print(torch.__version__.split('+')[0])")+$(python3 -c "import torch; print('cu'+torch.version.cuda.replace('.','') if torch.version.cuda else 'cpu')").html


Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 37.4 MB/s eta 0:00:00


In [ ]:
import torch
print(torch.__version__)


2.8.0+cu126


In [ ]:
# Install PyTorch Geometric dependencies for torch 2.8.0+cu126
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch-geometric -f https://data.pyg.org/whl/torch-2.8.0+cu126.html


Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 45.0 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.3 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 45.1 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html


In [ ]:
# Optional: pyg-lib for accelerated NeighborSampler
!pip install pyg-lib -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

Looking in links: https://data.pyg.org/whl/torch-2.8.0+cu126.html


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

# Same base dir you used before
BASE_DIR = "/content/drive/MyDrive/AML_Project_v2"

ACC_GRAPH_PT  = os.path.join(BASE_DIR, "graph_accounts_0p2.pt")
BANK_GRAPH_PT = os.path.join(BASE_DIR, "graph_banks_0p2.pt")

ACC_MAP_PKL   = os.path.join(BASE_DIR, "account_index_map_0p2.pkl")
BANK_MAP_PKL  = os.path.join(BASE_DIR, "bank_index_map_0p2.pkl")

SAMPLE_CSV    = os.path.join(BASE_DIR, "HI_Trans_0p2.csv")

# Outputs for MC-Dropout stage
ACC_EMB_MC_NPY   = os.path.join(BASE_DIR, "account_embeddings_sage_mc_0p2.npy")
BANK_EMB_MC_NPY  = os.path.join(BASE_DIR, "bank_embeddings_sage_mc_0p2.npy")
ACC_UNCERT_NPY   = os.path.join(BASE_DIR, "account_uncertainty_mc_0p2.npy")
BANK_UNCERT_NPY  = os.path.join(BASE_DIR, "bank_uncertainty_mc_0p2.npy")
MV_EMB_MC_NPY    = os.path.join(BASE_DIR, "node_embeddings_mv_sage_mc_0p2.npy")
ACC_UNSTABLE_NPY = os.path.join(BASE_DIR, "account_unstable_mask_mc_0p2.npy")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("BASE_DIR:", BASE_DIR)


Using device: cuda
BASE_DIR: /content/drive/MyDrive/AML_Project_v2


In [ ]:
import torch

# Load graphs from previous Stage B (explicitly allow full objects)
data_acc: Data = torch.load(ACC_GRAPH_PT, map_location=device, weights_only=False)
data_bank: Data = torch.load(BANK_GRAPH_PT, map_location=device, weights_only=False)

print("Account graph:", data_acc)
print("Bank graph   :", data_bank)

# Load mappings
account_to_idx = joblib.load(ACC_MAP_PKL)
bank_to_idx    = joblib.load(BANK_MAP_PKL)

num_acc_nodes  = data_acc.num_nodes
num_bank_nodes = data_bank.num_nodes

print("num_acc_nodes :", num_acc_nodes)
print("num_bank_nodes:", num_bank_nodes)


Account graph: Data(x=[331899, 8], edge_index=[2, 2031338], y=[331899], train_mask=[331899], val_mask=[331899], test_mask=[331899])
Bank graph   : Data(x=[19970, 8], edge_index=[2, 2031338], y=[19970], train_mask=[19970], val_mask=[19970], test_mask=[19970])
num_acc_nodes : 331899
num_bank_nodes: 19970


In [ ]:
class SAGEModelMC(nn.Module):
    def __init__(self, in_channels, hidden_dim=64, out_channels=2, dropout_p=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.lin   = nn.Linear(hidden_dim, out_channels)
        self.dropout_p = dropout_p

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)  # MC dropout layer 1

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)  # MC dropout layer 2

        return x

    def forward(self, x, edge_index):
        h = self.encode(x, edge_index)
        out = self.lin(h)
        return out, h


In [ ]:
def train_sage_mc(data: Data, hidden_dim=64, num_epochs=40, lr=1e-3, wd=1e-5):
    in_dim = data.num_features
    out_classes = int(data.y.max().item()) + 1

    model = SAGEModelMC(in_channels=in_dim, hidden_dim=hidden_dim, out_channels=out_classes, dropout_p=0.3).to(device)

    labels = data.y
    class_counts = torch.bincount(labels)
    print("Class counts:", class_counts.tolist())

    weights = class_counts.float()
    weights = weights.sum() / (2.0 * weights)  # simple inverse-frequency
    print("Class weights:", weights.tolist())

    criterion = nn.CrossEntropyLoss(weight=weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    def accuracy(logits, y_true):
        preds = logits.argmax(dim=1)
        correct = (preds == y_true).sum().item()
        return correct / y_true.numel()

    data = data.to(device)
    best_val_acc = 0.0
    best_state_dict = None

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()

        out, h = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits, _ = model(data.x, data.edge_index)
            train_acc = accuracy(logits[data.train_mask], data.y[data.train_mask])
            val_acc   = accuracy(logits[data.val_mask], data.y[data.val_mask])
            test_acc  = accuracy(logits[data.test_mask], data.y[data.test_mask])

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state_dict = model.state_dict()

        print(f"Epoch {epoch:03d} | Loss={loss.item():.4f} | "
              f"Train={train_acc:.4f} | Val={val_acc:.4f} | Test={test_acc:.4f}")

    print("Best val acc:", best_val_acc)
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
        print("Loaded best model weights.")

    return model.cpu()


In [ ]:
print("Training MC-Dropout GNN on ACCOUNT view...")
model_acc_mc = train_sage_mc(data_acc.clone(), hidden_dim=64, num_epochs=40)

print("\nTraining MC-Dropout GNN on BANK view...")
model_bank_mc = train_sage_mc(data_bank.clone(), hidden_dim=32, num_epochs=40)


Training MC-Dropout GNN on ACCOUNT view...
Class counts: [330241, 1658]
Class weights: [0.5025103092193604, 100.09017181396484]
Epoch 001 | Loss=30669776.0000 | Train=0.8404 | Val=0.8423 | Test=0.8416
Epoch 002 | Loss=23523678.0000 | Train=0.5872 | Val=0.5898 | Test=0.5908
Epoch 003 | Loss=23233022.0000 | Train=0.4685 | Val=0.4694 | Test=0.4695
Epoch 004 | Loss=16970534.0000 | Train=0.4392 | Val=0.4407 | Test=0.4406
Epoch 005 | Loss=18398938.0000 | Train=0.4814 | Val=0.4835 | Test=0.4833
Epoch 006 | Loss=22938398.0000 | Train=0.5125 | Val=0.5149 | Test=0.5145
Epoch 007 | Loss=21037734.0000 | Train=0.5425 | Val=0.5444 | Test=0.5437
Epoch 008 | Loss=16881104.0000 | Train=0.5950 | Val=0.5961 | Test=0.5946
Epoch 009 | Loss=17003674.0000 | Train=0.6810 | Val=0.6812 | Test=0.6791
Epoch 010 | Loss=14415523.0000 | Train=0.7051 | Val=0.7032 | Test=0.7021
Epoch 011 | Loss=15547147.0000 | Train=0.7041 | Val=0.7020 | Test=0.7013
Epoch 012 | Loss=13505661.0000 | Train=0.6414 | Val=0.6401 | Test=0.6

In [ ]:
def mc_sample_embeddings(model: SAGEModelMC, data: Data, num_samples=20):
    """
    MC-Dropout sampling:
      - keep dropout ON (model.train()) but no gradients
      - collect multiple logits & embeddings
    Returns:
      mean_emb: (num_nodes, hidden_dim)
      uncert:   (num_nodes,) = variance of class-1 prob across samples
    """
    model = model.to(device)
    data = data.to(device)

    all_logits = []
    all_embs   = []

    with torch.no_grad():
        for s in range(num_samples):
            model.train()  # important: dropout active
            out, h = model(data.x, data.edge_index)
            all_logits.append(out.unsqueeze(0).cpu().numpy())  # (1, N, C)
            all_embs.append(h.unsqueeze(0).cpu().numpy())      # (1, N, H)

    all_logits = np.concatenate(all_logits, axis=0)  # (S, N, C)
    all_embs   = np.concatenate(all_embs, axis=0)    # (S, N, H)

    mean_emb = all_embs.mean(axis=0)  # (N, H)

    logits_tensor = torch.tensor(all_logits)  # (S, N, C)
    probs_tensor  = torch.softmax(logits_tensor, dim=2)  # (S, N, C)
    probs_np      = probs_tensor.numpy()

    p1 = probs_np[:, :, 1]           # (S, N) prob of class 1
    var_p1 = p1.var(axis=0)          # (N,) variance across samples

    return mean_emb, var_p1


In [ ]:
print("MC sampling on ACCOUNT graph...")
mean_emb_acc, uncert_acc = mc_sample_embeddings(model_acc_mc, data_acc, num_samples=20)
print("Account mean_emb shape:", mean_emb_acc.shape)
print("Account uncertainty shape:", uncert_acc.shape)

print("\nMC sampling on BANK graph...")
mean_emb_bank, uncert_bank = mc_sample_embeddings(model_bank_mc, data_bank, num_samples=20)
print("Bank mean_emb shape:", mean_emb_bank.shape)
print("Bank uncertainty shape:", uncert_bank.shape)

np.save(ACC_EMB_MC_NPY,  mean_emb_acc)
np.save(BANK_EMB_MC_NPY, mean_emb_bank)
np.save(ACC_UNCERT_NPY,  uncert_acc)
np.save(BANK_UNCERT_NPY, uncert_bank)

print("\nSaved:")
print("  Account MC embeddings  →", ACC_EMB_MC_NPY)
print("  Bank MC embeddings     →", BANK_EMB_MC_NPY)
print("  Account uncertainty    →", ACC_UNCERT_NPY)
print("  Bank uncertainty       →", BANK_UNCERT_NPY)


MC sampling on ACCOUNT graph...
Account mean_emb shape: (331899, 64)
Account uncertainty shape: (331899,)

MC sampling on BANK graph...
Bank mean_emb shape: (19970, 32)
Bank uncertainty shape: (19970,)

Saved:
  Account MC embeddings  → /content/drive/MyDrive/AML_Project_v2/account_embeddings_sage_mc_0p2.npy
  Bank MC embeddings     → /content/drive/MyDrive/AML_Project_v2/bank_embeddings_sage_mc_0p2.npy
  Account uncertainty    → /content/drive/MyDrive/AML_Project_v2/account_uncertainty_mc_0p2.npy
  Bank uncertainty       → /content/drive/MyDrive/AML_Project_v2/bank_uncertainty_mc_0p2.npy


In [ ]:
# Mark top 10% highest-uncertainty accounts as unstable
threshold = np.quantile(uncert_acc, 0.90)
acc_is_unstable = (uncert_acc >= threshold).astype(int)  # 1 = unstable

print("Unstable accounts (top 10% by uncertainty):", acc_is_unstable.sum(), "/", len(acc_is_unstable))

np.save(ACC_UNSTABLE_NPY, acc_is_unstable)
print("Saved account unstable mask →", ACC_UNSTABLE_NPY)


Unstable accounts (top 10% by uncertainty): 43497 / 331899
Saved account unstable mask → /content/drive/MyDrive/AML_Project_v2/account_unstable_mask_mc_0p2.npy


In [ ]:
df_02 = pd.read_csv(SAMPLE_CSV)
df_02["Account"]   = df_02["Account"].astype(str)
df_02["Account.1"] = df_02["Account.1"].astype(str)
df_02["From Bank"] = df_02["From Bank"].astype(str)
df_02["To Bank"]   = df_02["To Bank"].astype(str)

# Most frequent From Bank per account
acc_from_counts = df_02.groupby(["Account", "From Bank"]).size().reset_index(name="count")
idx_max_from = acc_from_counts.groupby("Account")["count"].idxmax()
acc_primary_from = acc_from_counts.loc[idx_max_from, ["Account", "From Bank"]].set_index("Account")["From Bank"]

# Most frequent To Bank per account
acc_to_counts = df_02.groupby(["Account", "To Bank"]).size().reset_index(name="count")
idx_max_to = acc_to_counts.groupby("Account")["count"].idxmax()
acc_primary_to = acc_to_counts.loc[idx_max_to, ["Account", "To Bank"]].set_index("Account")["To Bank"]

print("Example mapping (Account -> From Bank):")
display(acc_primary_from.head())
print("\nExample mapping (Account -> To Bank):")
display(acc_primary_to.head())


Example mapping (Account -> From Bank):


,From Bank
Account,
100428660,70
1004286A8,70
1004286F0,70
100428738,70
100428780,70



Example mapping (Account -> To Bank):


,To Bank
Account,
100428660,10
1004286A8,22
1004286F0,3
100428738,15
100428780,16


In [ ]:
print("mean_emb_acc shape :", mean_emb_acc.shape)
print("mean_emb_bank shape:", mean_emb_bank.shape)

hidden_dim_acc  = mean_emb_acc.shape[1]
hidden_dim_bank = mean_emb_bank.shape[1]
final_dim = hidden_dim_acc + 2 * hidden_dim_bank

emb_mv_mc = np.zeros((num_acc_nodes, final_dim), dtype=np.float32)

for acc, acc_idx in account_to_idx.items():
    from_bank = acc_primary_from.get(acc, None)
    to_bank   = acc_primary_to.get(acc, None)

    acc_vec = mean_emb_acc[acc_idx]

    if (from_bank is not None) and (from_bank in bank_to_idx):
        fb_vec = mean_emb_bank[bank_to_idx[from_bank]]
    else:
        fb_vec = np.zeros(hidden_dim_bank, dtype=np.float32)

    if (to_bank is not None) and (to_bank in bank_to_idx):
        tb_vec = mean_emb_bank[bank_to_idx[to_bank]]
    else:
        tb_vec = np.zeros(hidden_dim_bank, dtype=np.float32)

    emb_mv_mc[acc_idx] = np.concatenate([acc_vec, fb_vec, tb_vec])

np.save(MV_EMB_MC_NPY, emb_mv_mc)
print("Saved multi-view MC embeddings →", MV_EMB_MC_NPY)
print("Final MV MC embedding shape:", emb_mv_mc.shape)


mean_emb_acc shape : (331899, 64)
mean_emb_bank shape: (19970, 32)
Saved multi-view MC embeddings → /content/drive/MyDrive/AML_Project_v2/node_embeddings_mv_sage_mc_0p2.npy
Final MV MC embedding shape: (331899, 128)
